# Task 1.1 — Preprocesamiento de ETTh1

In [1]:
import ssl
import urllib.request
import urllib.error
from pathlib import Path

import certifi
import numpy as np
import torch

torch.manual_seed(42)
print(f'PyTorch: {torch.__version__}')

PyTorch: 2.13.0


## 1. Descarga y carga de datos

In [2]:
URL = (
    'https://raw.githubusercontent.com/zhouhaoyi/'
    'ETDataset/main/ETT-small/ETTh1.csv'
)
DATA_PATH = Path('ETTh1.csv')

if not DATA_PATH.exists():
    try:
        urllib.request.urlretrieve(URL, DATA_PATH)
    except urllib.error.URLError as error:
        if DATA_PATH.exists():
            DATA_PATH.unlink()
        ssl_context = ssl.create_default_context(cafile=certifi.where())
        with urllib.request.urlopen(URL, context=ssl_context) as response:
            DATA_PATH.write_bytes(response.read())
        print(f'La descarga estándar falló ({error.reason}); se usó certifi.')
    print(f'Dataset descargado en {DATA_PATH.resolve()}')
else:
    print(f'Se reutiliza el dataset existente en {DATA_PATH.resolve()}')

data = np.genfromtxt(DATA_PATH, delimiter=',', skip_header=1)
OT = data[:, -1].astype(np.float32)
ALL = data[:, 1:].astype(np.float32)

assert OT.ndim == 1
assert ALL.ndim == 2 and ALL.shape[1] == 7
assert np.isfinite(OT).all() and np.isfinite(ALL).all()

print(f'Datos completos: {data.shape}')
print(f'OT: {OT.shape}')
print(f'ALL: {ALL.shape}')

La descarga estándar falló ([SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1028)); se usó certifi.
Dataset descargado en /Users/nilsmurallesmorales/universidad/octavosemestre/deep/lab7/ETTh1.csv
Datos completos: (17420, 8)
OT: (17420,)
ALL: (17420, 7)


## 2. Normalización de `OT`

In [3]:
OT_tensor = torch.from_numpy(OT)

mu = OT_tensor.mean()
sigma = OT_tensor.std(correction=0)
assert sigma > 0, 'La desviación estándar debe ser positiva.'

OT_norm = (OT_tensor - mu) / sigma

print(f'mu    = {mu.item():.6f}')
print(f'sigma = {sigma.item():.6f}')
print(f'Media normalizada: {OT_norm.mean().item():.7f}')
print(f'Desv. normalizada: {OT_norm.std(correction=0).item():.7f}')

mu    = 13.324672
sigma = 8.566700
Media normalizada: -0.0000000
Desv. normalizada: 1.0000000


## 3. Split temporal contiguo 60/20/20

In [4]:
T = OT_norm.numel()
train_end = int(0.60 * T)
val_end = int(0.80 * T)

train_series = OT_norm[:train_end].clone()
val_series = OT_norm[train_end:val_end].clone()
test_series = OT_norm[val_end:].clone()

assert train_series.numel() + val_series.numel() + test_series.numel() == T
assert torch.equal(torch.cat((train_series, val_series, test_series)), OT_norm)

print(f'Total:         {T:5d} observaciones')
print(f'Entrenamiento: {train_series.numel():5d} ({train_series.numel()/T:.2%})')
print(f'Validación:    {val_series.numel():5d} ({val_series.numel()/T:.2%})')
print(f'Prueba:        {test_series.numel():5d} ({test_series.numel()/T:.2%})')
print(f'Índices: train=[0, {train_end}), val=[{train_end}, {val_end}), test=[{val_end}, {T})')

Total:         17420 observaciones
Entrenamiento: 10452 (60.00%)
Validación:     3484 (20.00%)
Prueba:         3484 (20.00%)
Índices: train=[0, 10452), val=[10452, 13936), test=[13936, 17420)


## 4. Construcción de ventanas

In [5]:
def make_windows(series: torch.Tensor, L: int, H: int) -> tuple[torch.Tensor, torch.Tensor]:
    if series.ndim != 1:
        raise ValueError(f'series debe ser 1D; forma recibida: {tuple(series.shape)}')
    if not isinstance(L, int) or not isinstance(H, int) or L <= 0 or H <= 0:
        raise ValueError('L y H deben ser enteros positivos.')
    if series.numel() < L + H:
        raise ValueError(
            f'Serie insuficiente: se requieren al menos L+H={L+H} valores, '
            f'pero hay {series.numel()}.'
        )

    windows = series.unfold(dimension=0, size=L + H, step=1)
    X = windows[:, :L].contiguous()
    y = windows[:, L + H - 1].contiguous()
    return X, y

### Prueba pequeña de indexación

In [6]:
toy = torch.arange(8, dtype=torch.float32)
X_toy, y_toy = make_windows(toy, L=3, H=2)

assert torch.equal(X_toy[0], torch.tensor([0.0, 1.0, 2.0]))
assert y_toy[0].item() == 4.0
assert X_toy.shape == (4, 3) and y_toy.shape == (4,)

print('X de prueba:\n', X_toy)
print('y de prueba:', y_toy)

X de prueba:
 tensor([[0., 1., 2.],
        [1., 2., 3.],
        [2., 3., 4.],
        [3., 4., 5.]])
y de prueba: tensor([4., 5., 6., 7.])


## 5. Ventanas para $L=96$ y $H\in\{24,48\}$

In [7]:
L = 96
HORIZONS = (24, 48)
series_by_split = {
    'train': train_series,
    'val': val_series,
    'test': test_series,
}

datasets = {}
for H in HORIZONS:
    datasets[H] = {}
    print(f'\nHorizonte H={H}')
    for split_name, split_series in series_by_split.items():
        X, y = make_windows(split_series, L=L, H=H)
        datasets[H][split_name] = (X, y)

        expected_n = split_series.numel() - L - H + 1
        assert X.shape == (expected_n, L)
        assert y.shape == (expected_n,)
        assert X.dtype == torch.float32 and y.dtype == torch.float32
        print(f'  {split_name:5s}: X={tuple(X.shape)}, y={tuple(y.shape)}')


Horizonte H=24
  train: X=(10333, 96), y=(10333,)
  val  : X=(3365, 96), y=(3365,)
  test : X=(3365, 96), y=(3365,)

Horizonte H=48
  train: X=(10309, 96), y=(10309,)
  val  : X=(3341, 96), y=(3341,)
  test : X=(3341, 96), y=(3341,)


## 6. Acceso y desnormalización

In [8]:
X_train_24, y_train_24 = datasets[24]['train']
X_val_24, y_val_24 = datasets[24]['val']
X_test_24, y_test_24 = datasets[24]['test']

X_train_48, y_train_48 = datasets[48]['train']
X_val_48, y_val_48 = datasets[48]['val']
X_test_48, y_test_48 = datasets[48]['test']

y_train_24_celsius = y_train_24 * sigma + mu

first_target_index = L + 24 - 1
assert torch.allclose(y_train_24_celsius[0], OT_tensor[first_target_index])

print('Primer objetivo normalizado:', y_train_24[0].item())
print('Primer objetivo en °C:', y_train_24_celsius[0].item())

Primer objetivo normalizado: 2.0002248287200928
Primer objetivo en °C: 30.459999084472656
